In [1]:
import numpy as np
from scipy import stats
import pandas as pd

In [13]:
params = {
    'Container': {'lam': 5.5812, 'mu': 9.6069, 'sigma': 0.3493},
    'Tanker': {'lam': 2.2299, 'mu': 10.9119, 'sigma': 0.2963},
    'Bulk': {'lam': 2.2947, 'mu': 10.4989, 'sigma': 0.3806},
}

days_in_year = 365
nr_runs = 10000
rng = np.random.default_rng(42)
def simPoissonProcess(lam, t, rng):
    intArrivalTimsDist = stats.expon(scale=1/lam)
    Nt = 0
    time = intArrivalTimsDist.rvs(random_state=rng)
    while time < t:
        Nt += 1
        time += intArrivalTimsDist.rvs(random_state=rng)
    return Nt


results_for_all_runs = {k: np.zeros(nr_runs) for k in 
                        ['NC', 'NT', 'NB', 'N', 'SC', 'ST', 'SB', 'S']}

for r in range(nr_runs): 
    NC = simPoissonProcess(params['Container']['lam'], days_in_year, rng)
    NT = simPoissonProcess(params['Tanker']['lam'], days_in_year, rng)
    NB = simPoissonProcess(params['Bulk']['lam'], days_in_year, rng)

    N = NC + NT + NB

    SC = stats.lognorm(s=params['Container']['sigma'], scale=np.exp(params['Container']['mu'])).rvs(size=NC, random_state=rng).sum()
    ST = stats.lognorm(s=params['Tanker']['sigma'], scale=np.exp(params['Tanker']['mu'])).rvs(size=NT, random_state=rng).sum()
    SB = stats.lognorm(s=params['Bulk']['sigma'], scale=np.exp(params['Bulk']['mu'])).rvs(size=NB, random_state=rng).sum()

    S = SC + ST + SB

    results_for_all_runs['NC'][r],results_for_all_runs['NT'][r], results_for_all_runs['NB'][r] = NC, NT, NB
    results_for_all_runs['N'][r] = N
    results_for_all_runs['SC'][r], results_for_all_runs['ST'][r], results_for_all_runs['SB'][r] = SC, ST, SB
    results_for_all_runs['S'][r] = S

# Table 2
rows = []
for key in ['NC', 'NT', 'NB', 'N', 'SC', 'ST', 'SB', 'S']:
    x = results_for_all_runs[key]
    rows.append([key, x.mean(), x.std(ddof=1), np.percentile(x, 95)])

table2 = pd.DataFrame(rows, columns=['Quantity', 'Mean', 'Standard deviation', '95% quantile'])


    



In [14]:
pd.DataFrame(results_for_all_runs).to_csv('basic_results.csv', index=False)
table2.to_csv('basic_table2.csv', index=False)
print("Saved.")


Saved.


In [ ]:

T = 365
types = ['Container', 'Tanker', 'Bulk']

# Theoretical values from compound Poisson formulas
theo = {}

# Arrival counts: E[N_i] = λ_i T,  Var[N_i] = λ_i T
for t in types:
    lam = params[t]['lam']
    theo[f'N_{t[0]}(T)_mean'] = lam * T
    theo[f'N_{t[0]}(T)_std']  = np.sqrt(lam * T)

# Total arrivals: sum of independent Poissons
lam_total = sum(params[t]['lam'] for t in types)
theo['N(T)_mean'] = lam_total * T
theo['N(T)_std']  = np.sqrt(lam_total * T)

# Cargo per type: E[S_i] = λ_i T · E[X], Var[S_i] = λ_i T · E[X²]
def EX(mu, sigma):
    return np.exp(mu + sigma**2 / 2)

def EX2(mu, sigma):
    return np.exp(2 * mu + 2 * sigma**2)

for t in types:
    lam = params[t]['lam']
    mu  = params[t]['mu']
    sg  = params[t]['sigma']
    theo[f'S_{t[0]}(T)_mean'] = lam * T * EX(mu, sg)
    theo[f'S_{t[0]}(T)_std']  = np.sqrt(lam * T * EX2(mu, sg))

# Total cargo: sum of independent compound Poissons
theo['S(T)_mean'] = sum(theo[f'S_{t[0]}(T)_mean'] for t in types)
theo['S(T)_std']  = np.sqrt(sum(theo[f'S_{t[0]}(T)_std']**2 for t in types))

# Simulated values from results dict 
sim = {k: results_for_all_runs[k].mean() for k in results_for_all_runs}
sim_std = {k: results_for_all_runs[k].std(ddof=1) for k in results_for_all_runs}

# Build the comparison table (with percentage)
rows = []
mapping = [
    ('E[N_C(T)]',   'NC', 'N_C(T)_mean'),
    ('Std[N_C(T)]', 'NC', 'N_C(T)_std'),
    ('E[N_T(T)]',   'NT', 'N_T(T)_mean'),
    ('Std[N_T(T)]', 'NT', 'N_T(T)_std'),
    ('E[N_B(T)]',   'NB', 'N_B(T)_mean'),
    ('Std[N_B(T)]', 'NB', 'N_B(T)_std'),
    ('E[N(T)]',     'N',  'N(T)_mean'),
    ('Std[N(T)]',   'N',  'N(T)_std'),
    ('E[S_C(T)]',   'SC', 'S_C(T)_mean'),
    ('Std[S_C(T)]', 'SC', 'S_C(T)_std'),
    ('E[S_T(T)]',   'ST', 'S_T(T)_mean'),
    ('Std[S_T(T)]', 'ST', 'S_T(T)_std'),
    ('E[S_B(T)]',   'SB', 'S_B(T)_mean'),
    ('Std[S_B(T)]', 'SB', 'S_B(T)_std'),
    ('E[S(T)]',     'S',  'S(T)_mean'),
    ('Std[S(T)]',   'S',  'S(T)_std'),
]

for label, sim_key, theo_key in mapping:
    theoretical = theo[theo_key]
    simulated   = sim[sim_key] if 'mean' in theo_key else sim_std[sim_key]
    diff_pct    = (simulated - theoretical) / theoretical * 100
    rows.append([label, theoretical, simulated, diff_pct])

validation = pd.DataFrame(rows,
    columns=['Quantity', 'Theoretical', 'Simulated', 'Difference (%)'])

print(validation.to_string(index=False, float_format='{:,.2f}'.format))

   Quantity    Theoretical      Simulated  Difference (%)
  E[N_C(T)]       2,037.14       2,037.66            0.03
Std[N_C(T)]          45.13          45.63            1.10
  E[N_T(T)]         813.91         813.28           -0.08
Std[N_T(T)]          28.53          28.85            1.14
  E[N_B(T)]         837.57         837.84            0.03
Std[N_B(T)]          28.94          28.88           -0.20
    E[N(T)]       3,688.62       3,688.78            0.00
  Std[N(T)]          60.73          60.74            0.02
  E[S_C(T)]  32,191,287.70  32,197,819.74            0.02
Std[S_C(T)]     758,092.79     765,562.56            0.99
  E[S_T(T)]  46,625,163.38  46,593,497.49           -0.07
Std[S_T(T)]   1,707,636.38   1,727,383.40            1.16
  E[S_B(T)]  32,665,422.14  32,681,918.79            0.05
Std[S_B(T)]   1,213,483.97   1,206,336.38           -0.59
    E[S(T)] 111,481,873.22 111,473,236.02           -0.01
  Std[S(T)]   2,227,839.77   2,224,293.73           -0.16


In [10]:
# Table of arrival share VS tonnage share 

arrivals = {
    'Container': results_for_all_runs['NC'].mean(),
    'Tanker': results_for_all_runs['NT'].mean(),
    'Bulk': results_for_all_runs['NB'].mean(),
}

tonnage = {
    'Container': results_for_all_runs['SC'].mean(),
    'Tanker': results_for_all_runs['ST'].mean(),
    'Bulk': results_for_all_runs['SB'].mean(),
}

total_arrivals = sum(arrivals.values())
total_tonnage = sum(tonnage.values())

share_compare = pd.DataFrame({
    'Vessel type': list(arrivals.keys()),
    'Arrivals': [arrivals[t] for t in arrivals],
    'Tonnage(Million tons)': [tonnage[t] / 1e6 for t in tonnage],
})

share_compare['Arrival share(%)'] = (share_compare['Arrivals'] / total_arrivals * 100).round(2)
share_compare['Tonnage share(%)'] = (share_compare['Tonnage(Million tons)'] / (total_tonnage/1e6) * 100).round(2)

print(share_compare.to_string(index=False, float_format='{:,.2f}'.format))

Vessel type  Arrivals  Tonnage(Million tons)  Arrival share(%)  Tonnage share(%)
  Container  2,037.66                  32.20             55.24             28.88
     Tanker    813.28                  46.59             22.05             41.80
       Bulk    837.84                  32.68             22.71             29.32


In [2]:
import time as _time
file_rotterdamPortData = r'..\data\rotterdam_dataset_16.xlsx'
wd = pd.read_excel(file_rotterdamPortData)

In [3]:
types = ['Container', 'Tanker', 'Bulk']

# Actual exact counts computed from the dataset date range
date_range = pd.date_range(wd['Arrival_Date'].min(),
                           wd['Arrival_Date'].max(),
                           freq='D')
days_per_month = date_range.to_series().groupby(date_range.month).size()

print("Days per calendar month in the observation window:")
print(days_per_month)
print()

Days per calendar month in the observation window:
1     93
2     85
3     93
4     90
5     93
6     90
7     93
8     93
9     90
10    93
11    90
12    93
dtype: int64



In [4]:
# Monthly arrival counts per type
monthly_count = {t: wd[wd['Vessel_Type'] == t]
                    .groupby(wd['Arrival_Date'].dt.month)
                    .size()
                 for t in types}

# Monthly rates λ_m = (total arrivals in month m) / (total days in month m) 
lam_month = {t: {m: monthly_count[t].get(m, 0) / days_per_month[m]
                 for m in range(1, 13)}
             for t in types}

print("Monthly rates (vessels/day):")
print(pd.DataFrame(lam_month).round(4))
print()

Monthly rates (vessels/day):
    Container  Tanker    Bulk
1      4.4946  2.0323  2.0323
2      4.8471  1.9529  2.0824
3      5.4839  2.2258  2.3011
4      5.1222  2.1556  2.0667
5      6.4194  2.4731  2.9032
6      6.2667  2.9111  2.5222
7      6.8817  2.5484  2.4839
8      6.6452  2.2796  2.5806
9      5.2889  2.0333  2.1556
10     5.5161  2.1290  2.3978
11     5.1889  2.2333  2.2444
12     4.7419  1.7742  1.7419



In [5]:
# Sanity check: weighted mean match constant lambda
params = {
    'Container': {'lam': 5.5812, 'mu': 9.6069, 'sigma': 0.3493},
    'Tanker': {'lam': 2.2299, 'mu': 10.9119, 'sigma': 0.2963},
    'Bulk': {'lam': 2.2947, 'mu': 10.4989, 'sigma': 0.3806},
}

for t in types:
    weighted_mean = (sum(lam_month[t][m] * days_per_month[m] for m in range(1, 13))
                     / days_per_month.sum())
    print(f"{t}: weighted mean = {weighted_mean:.4f}, "
          f"constant λ = {params[t]['lam']:.4f}")
print()

Container: weighted mean = 5.5812, constant λ = 5.5812
Tanker: weighted mean = 2.2299, constant λ = 2.2299
Bulk: weighted mean = 2.2947, constant λ = 2.2947



In [10]:
def simSeasonalPoisson(lam_by_month, rng):
    month_days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    start_date_of_month = np.cumsum([0] + month_days[:-1])

    Nt = 0
    time = 0.0
    while True: 
        # Find month index
        month_idx = np.searchsorted(start_date_of_month, time, side='right') - 1
        lam = lam_by_month[month_idx + 1]

        gap = rng.exponential(scale=1/lam)
        time += gap
        if time >= 365: 
            break
        Nt += 1
    return Nt

nr_runs = 1000

results_seasonal = {k: np.zeros(nr_runs) for k in ['NC','NT','NB','N','SC','ST','SB','S']} 
t0 = _time.time()
for r in range(nr_runs):
    NC = simSeasonalPoisson(lam_month['Container'], rng)
    NT = simSeasonalPoisson(lam_month['Tanker'], rng)
    NB = simSeasonalPoisson(lam_month['Bulk'], rng)
    N  = NC + NT + NB

    SC = rng.lognormal(mean=params['Container']['mu'], sigma=params['Container']['sigma'], size=NC).sum()
    ST = rng.lognormal(mean=params['Tanker']['mu'],    sigma=params['Tanker']['sigma'],    size=NT).sum()
    SB = rng.lognormal(mean=params['Bulk']['mu'],      sigma=params['Bulk']['sigma'],      size=NB).sum()
    S  = SC + ST + SB

    results_seasonal['NC'][r], results_seasonal['NT'][r], results_seasonal['NB'][r] = NC, NT, NB
    results_seasonal['N'][r]  = N
    results_seasonal['SC'][r], results_seasonal['ST'][r], results_seasonal['SB'][r] = SC, ST, SB
    results_seasonal['S'][r]  = S



In [15]:
results_for_all_runs = {col: pd.read_csv('basic_results.csv')[col].values
                        for col in pd.read_csv('basic_results.csv').columns}
table2 = pd.read_csv('basic_table2.csv')

rows = []
for key in ['N', 'S']:
    xb = results_for_all_runs[key]
    xs = results_seasonal[key]
    delta_std = (xs.std(ddof=1) - xb.std(ddof=1)) / xb.std(ddof=1) * 100
    rows.append([key,
                 xb.mean(), xb.std(ddof=1),
                 xs.mean(), xs.std(ddof=1),
                 delta_std])

seasonal_table = pd.DataFrame(rows, columns=[
    'Quantity',
    'Basic Mean', 'Basic Std',
    'Seasonal Mean', 'Seasonal Std',
    'Δ Std (%)'])

print()
print(seasonal_table.round(2).to_string(index=False))


Quantity   Basic Mean  Basic Std  Seasonal Mean  Seasonal Std  Δ Std (%)
       N      3688.78      60.74        3689.72         60.60      -0.23
       S 111473236.02 2224293.73   111469259.44    2177184.94      -2.12
